# kappa and the square-loop trap (H-loop): a test that failed

## Question and why it matters
Why does a longer opening same-letter run ("lead") raise kappa = (B/S)/(P/A) at fixed area? Hypothesis H-loop: an all-R lead traces 2x2 loops on young (colour < lead) frontier cells, so the ant spends more frontier time circling at the boundary. Parent: `../2026-10-kappa-leadrun-fresh/` (lead effect +0.152 ln-kappa per step on 176 stationary fresh rules). See `README.md` (registered predictions P1-P4, C1-C3) and `NOTEBOOK.md`.

## Assumptions and definitions
- Loop step: the ant's position equals its position 4 steps earlier and the last four turns are the same letter (a 2x2 lap). Lead-colour loop: all four cells read had colour < lead. Boundary/interior: the probe's boundary step. Window: probe-CSV k = 26 (t_end 2^26), the window of the parent's kappa.
- Frozen before any loop data (NOTEBOOK 21:58): the deciding mediator is the *interior-only* loop-step rate (a boundary-based mediator is nearly an identity with B/S); pass = b1/b0 <= 0.5, ratio CI upper < 0.8, g > 0; reliability rule ICC(1) >= 0.5.
- **This cohort is the one in which the lead effect was found**: mediation here is legitimate but not a replication.

## Provenance

In [1]:
import hashlib, subprocess, datetime, json
from pathlib import Path
import numpy as np, polars as pl
SEED = 20261016; np.random.seed(SEED)
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    assert root != root.parent, "repo root not found"; root = root.parent
R = root / "experiments/archive/2026-10-kappa-square-loop/results"
for f in ["analysis.txt", "analysis.json", "loops_per_rule.csv", "c3_pairs.csv", "run_cohort.log", "run_mutants.log"]:
    print(f"{f:20s} {hashlib.sha256((R / f).read_bytes()).hexdigest()[:16]}")
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=root, capture_output=True, text=True).stdout.strip(), "seed", SEED, datetime.date.today())
print((R / "run_cohort.log").read_text())

analysis.txt         785f72f4c26b7155
analysis.json        82d525a6c424ca39
loops_per_rule.csv   5bf73d400e0384f9
c3_pairs.csv         56ee5fec6972a828
run_cohort.log       a3fa7d5f83f0f892
run_mutants.log      1d7f63db457a0e9e
git d416073 seed 20261016 2026-10-16
git ce87387 2026-10-16T21:51:33+13:00 cmd: kappa_square_loop_sampler 27 8192 20 < experiments/2026-10-16-kappa-leadrun-fresh/results/specs.txt
2400 specs
0
200
400
600
800
1000
1200
1400
1600
1800
2000
2200
finished 2026-10-16T22:07:53+13:00
797895b9ccf77984ddf1161cb5ca396842a1171b172ac70ddedd81f13db650f9  target/release/examples/kappa_square_loop_sampler
cceddcd486199f2382ab161b514348ee3159ab30c97dc5b018c33ab86bc98ab6  experiments/2026-10-16-kappa-square-loop/dumps/loops_cohort.csv



## Method
Rust sampler `examples/kappa_square_loop_sampler.rs` (copy of the `probe_walker` stepping loop plus loop counters), tested against `probe_walker` and a store-the-whole-path reference (`tests/kappa_square_loop.rs`: 300 ant + 256 walker proptest cases, blank-board traps, PersistentWalk against exact q^4+(1-q)^4, rotor = exactly 0, 1 vs 6 threads byte-identical). Cohort: 300 rules x seeds 601..608 to 2^27; analysis `scripts/analyse_loops.py` (run once). This notebook reads only the committed outputs.

## Results 1: P2 (loops vs lead) and retention

In [2]:
txt = (R / "analysis.txt").read_text()
print(txt.split("=== Reliability")[0])

# cmd: uv run experiments/2026-10-16-kappa-square-loop/scripts/analyse_loops.py  git: f26b6ef (working tree may be dirty)  seed 20261016  bootstraps 2000  window k=26
IDENTITY vs parent dumps/probe.csv (probe-CSV k = log2(t_end) >= 21): rows parent 14509, sampler 14509, all of area/perimeter/new_cells/boundary_steps/escaped equal: True
parent table reproduced: 183 kept rules, 176 stationary, ln kappa equal to fresh_rules.csv to float precision
retention (of 50 sampled per lead): L1: kept 37, stationary 34; L2: kept 32, stationary 32; L3: kept 35, stationary 34; L4: kept 31, stationary 31; L5: kept 28, stationary 26; L6: kept 20, stationary 19
stationary fresh rules n=176; per lead 1:34, 2:32, 3:34, 4:31, 5:26, 6:19

=== P2: loop-step fraction (all loop steps / steps, k=26) by lead; retention beside it ===
lead  n  kept/50 stat/50  frac_all   frac_lead_colour  M_int(interior rate)  lead_colour_share_of_loop_steps  mean_resid_lnk(vs spline-only fit)
  1  34    37    34   0.15865   0.0000

**Shows:** the total loop-step fraction is large (16-26% of all steps) but is not monotone in lead (lead 3 < lead 2) and its within-area-tertile Spearman with lead is +0.50, +0.40, +0.17 (registered: > 0.5 in each). P2 failed, so by the registered logic H-loop is dead at this scale. **Does not show:** that the lead-colour laps are absent: they rise monotonically with lead (1e-5 to 1.6% of steps) but are too rare to matter in this window.

## Results 2: reliability, P3 and C1 mediation

In [3]:
print("=== Reliability" + txt.split("=== Reliability")[1].split("=== C2")[0])

=== Reliability of the deciding mediator M_int (frozen rule: ICC(1) across core-band runs < 0.5 => P3/C1/C2 uninformative) ===
  M_int: ICC(1) single run = 1.000 (between-rule sd 0.08012, within-rule (between-seed) sd 0.00020, 176 rules, mean runs/rule 7.91); reliability of the rule mean (Spearman-Brown) = 1.000
  frac_all: ICC(1) = 1.000
  M_int_lead: ICC(1) = 1.000
  M_int_other: ICC(1) = 1.000
  ent_int: ICC(1) = 1.000
  reliability rule: ICC >= 0.5, mediation test is informative

=== P3: ln kappa ~ spline(ln A22) + b*min(lead,6) [+ g*mediator]; census-quantile spline knots, OLS on the stationary fresh rules ===
  retention (of 50 sampled per lead): L1: kept 37, stationary 34; L2: kept 32, stationary 32; L3: kept 35, stationary 34; L4: kept 31, stationary 31; L5: kept 28, stationary 26; L6: kept 20, stationary 19
  Pass (frozen): b1/b0 <= 0.5 and ratio CI upper < 0.8 and g > 0 with CI excluding 0. DECIDING mediator = M_int (interior-only loop-step rate). Others are accounting/robust

**Shows:** the loop rate is a nearly deterministic property of the rule (ICC 1.0), so the null mediation is not a noise artefact. Adding the interior loop rate leaves the lead coefficient at 0.957 of its value (needed <= 0.5), also on leads 1-4 (0.98), under log/rank transforms, and for the lead-colour loops alone the sign is wrong (ratio 1.13). Only the boundary-based loop rate (accounting: nearly B/S by construction) removes part of it (0.73). **Does not show:** that no loops of another shape (period 6/8) matter; the lead effect itself remains unexplained.

## Results 3: P4 (boundary concentration) and C3 (mutants)

In [4]:
print("=== C2" + txt.split("=== C2")[1])
p = pl.read_csv(R / "c3_pairs.csv")
print(p.group_by("arm").agg(n=pl.len(), mean_dM=pl.col("dM").mean(), mean_d=pl.col("d").mean()).sort("arm"))

=== C2 (identity caveat) ===
  C2 holds iff the interior-only M_int passes P3: False. Boundary-loop mediator (accounting) ratio = +0.727; all-loop accounting ratio = +0.955; interior entry-rate ratio = +1.019

=== P4: P(loop | boundary step) / P(loop | interior step), k=26, per rule (pooled over its core runs), stationary rules ===
  median ratio 0.776 CI [0.579,0.957]; quartiles [0.189, 1.306]; by lead median: 1:0.13, 2:0.72, 3:0.76, 4:0.74, 5:1.18, 6:1.27
  P4 FAILED (median >= 2). Partly accounting by construction (the loop step is classed by its start cell).

=== C3 (mutation link) ===
  pairs (arms E+S, both kept): n=154 (E 81, S 73); Spearman(d M_int, d ln kappa) = +0.218 CI [+0.061,+0.364]; accounting Spearman(d frac_all, d) = +0.225
    arm E: n=81, mean dM_int = +0.03019, mean d ln kappa = -0.062, Spearman = +0.186
    arm S: n=73, mean dM_int = -0.01028, mean d ln kappa = -0.198, Spearman = +0.137
  C3 FAILED (Spearman > 0.3; not powered)

shape: (2, 4)
┌─────┬─────┬─────────

**Shows:** loops are not concentrated on boundary steps (median ratio 0.78, registered >= 2) and the mutant link is weak (Spearman 0.22 < 0.3, 154 pairs, unpowered). **Does not show:** anything causal; the surgery experiment was not run because P2 failed.

In [5]:
t = pl.read_csv(R / "loops_per_rule.csv", comment_prefix="#").filter(pl.col("stationary"))
print(t.group_by("lead").agg(n=pl.len(), frac_all=pl.col("frac_all").mean(), frac_lead=pl.col("frac_lead").mean(), M_int=pl.col("M_int").mean(), P4=pl.col("p4").median()).sort("lead"))
a = json.loads((R / "analysis.json").read_text())
assert not a["P2"]["held"] and not a["P3"]["pass_main"] and not a["C1"]["held"] and not a["P4"]["held"] and not a["C3"]["held"]
print("verdicts agree with README: P2, P3, C1, C2, P4, C3 all FAILED; reliability ICC", round(a["reliability"]["icc"], 3))

shape: (6, 6)
┌──────┬─────┬──────────┬───────────┬──────────┬──────────┐
│ lead ┆ n   ┆ frac_all ┆ frac_lead ┆ M_int    ┆ P4       │
│ ---  ┆ --- ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ i64  ┆ u32 ┆ f64      ┆ f64       ┆ f64      ┆ f64      │
╞══════╪═════╪══════════╪═══════════╪══════════╪══════════╡
│ 1    ┆ 34  ┆ 0.158646 ┆ 0.000013  ┆ 0.159708 ┆ 0.129093 │
│ 2    ┆ 32  ┆ 0.183877 ┆ 0.000235  ┆ 0.184262 ┆ 0.723975 │
│ 3    ┆ 34  ┆ 0.166073 ┆ 0.000988  ┆ 0.165995 ┆ 0.760486 │
│ 4    ┆ 31  ┆ 0.179093 ┆ 0.003927  ┆ 0.179088 ┆ 0.743242 │
│ 5    ┆ 26  ┆ 0.201632 ┆ 0.010413  ┆ 0.200993 ┆ 1.177235 │
│ 6    ┆ 19  ┆ 0.256702 ┆ 0.016331  ┆ 0.255903 ┆ 1.273667 │
└──────┴─────┴──────────┴───────────┴──────────┴──────────┘
verdicts agree with README: P2, P3, C1, C2, P4, C3 all FAILED; reliability ICC 1.0


## Caveats and controls
- Same cohort that produced the lead effect; retention falls with lead (kept 37, 32, 35, 31, 28, 20 of 50), so leads 5-6 are the most selected; P3 on leads 1-4 gives the same answer.
- Window labelling: the probe CSV's k is log2(t_end); the analysis uses that convention (NOTEBOOK 22:36).
- Positive controls: the sampler reproduces the parent's probe rows on all 14509 rows and the exact PersistentWalk values; negative control: rotor has exactly zero loop steps (a theorem).
- Registered thresholds are my design choices; none were changed after seeing data. Mutant C3 is unpowered.

## What this changes / next steps
H-loop (as stated) is rejected for this cohort and window. Remaining open: mixed-turn period-6/8 loops, loops in the opening transient, a different outcome than B/S (e.g. kappa's P/A factor), and whether the lead effect is a rule-composition effect the loop statistic only proxies.

## Open questions
Why is the generic same-turn loop rate 16-26% at every lead and nearly seed-independent? What does carry the lead effect, given that the interior loop rate, lead-colour laps and (by earlier work) R-count do not?